# Lectures 7-15: SDK Foundations in Colab

This notebook combines the Section 3 hands-on lessons into one Colab-ready workflow:

- Lecture 7: Installing and running the SDK
- Lecture 8: Validating and explaining standards files
- Lecture 9: Using ODPV vocabulary helpers
- Lecture 10: Working with local LLMs
- Lecture 11: Working with online LLM providers
- Lecture 12: Generating ODPS data products
- Lecture 13: Generating ODPC fragments
- Lecture 14: Working with ODPG graphs
- Lecture 15: Working with ODPC catalogs

Run the cells from top to bottom. Some generation cells require an `ANTHROPIC_API_KEY` stored in Colab secrets. The validation, vocabulary, graph, and catalog cells can still run without an LLM key.


### What We Just Covered

Before opening the hands-on workbook, we introduced the SDK as the practical tooling layer for the Open Data Products standards family. The SDK helps turn standards from static documents into workflows for validation, generation, explanation, catalogs, graphs, HTML review pages, and agent-ready YAML.

In this notebook, we now move from that mental model into the first executable SDK exercises.

# Lecture 7: Installing and Running the SDK

This section prepares Colab, clones the course exercise repository, installs the SDK, and creates one workspace folder for Section 3 outputs.


## Clone The Course Exercises

The course repository contains the sample files used by the lessons. In this workbook, we run the exercises directly inside the cloned lecture folders under:

```text
/content/odps-sdk-course-exercises/
```

Generated files are written beside each lesson's sample files in folders such as `products/`, `fragments/`, or `output/`.

In [12]:
%cd /content
!rm -rf /content/odps-sdk-course-exercises
!git clone https://github.com/Open-Data-Product-Initiative/odps-sdk-course-exercises.git
%cd /content/odps-sdk-course-exercises

/content
Cloning into 'odps-sdk-course-exercises'...
remote: Enumerating objects: 349, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 349 (delta 132), reused 279 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (349/349), 118.62 KiB | 13.18 MiB/s, done.
Resolving deltas: 100% (132/132), done.
/content/odps-sdk-course-exercises


## Install The SDK

Install specific released version of the SDK into the Colab runtime. **Re-run this cell if Colab restarts.**


In [2]:
!python -m pip install --upgrade open-data-products==0.2.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.9/320.9 kB 17.6 MB/s eta 0:00:00


## Check The SDK Command

The SDK CLI is named `open-data-products`.


In [3]:
!open-data-products --version
!open-data-products --help


open-data-products 0.2.4
usage: open-data-products [-h] [-V] COMMAND ...

Validate, inspect, and expose Open Data Product family artifacts.

positional arguments:
  COMMAND
    validate            Validate a document
    explain             Explain a document
    refs                List document references
    resources           List SDK resources
    summary             Lightweight artifact reference for a document
    config              Show or copy SDK config templates
    generate            Generate selected YAML artifacts with configured LLMs
    odpc-summary        Summarize an ODPC catalog
    odpc-build          Build an ODPC catalog from fragments
    odpc-search         Search ODPC catalog object guidance
    odpc-artifacts      Generate or check derived ODPC catalog artifacts
    odpv-summary        Summarize an ODPV vocabulary
    odpv-search         Search ODPV vocabulary terms
    odpv-resolve        Resolve text or aliases to a canonical ODPV term
    odpv-explain   

## Run A Simple Machine-Readable Command

`manifest --json` proves the package imports and the CLI can render structured output. JSON is most useful for scripts, automation, and agents.


### Before You Run This

Many SDK commands can produce either human-readable output or structured JSON. In this quick check, `--json` shows the SDK as an automation tool: scripts and AI agents can read the same command output that a person can inspect in the notebook.

What the code does:

- Runs `manifest` to ask the SDK what capabilities are available.
- Uses `--json` so the output is machine-readable.
- Pipes the JSON through `python -m json.tool` to make it easier to read.
- Shows only the first part of the output with `head -60` so the notebook stays compact.


In [4]:
!open-data-products manifest \
  --json \
  | python -m json.tool \
  | head -60


{
    "name": "open-data-products",
    "description": "Validate, explain, traverse, and search Open Data Products documents (ODPS, ODPC, ODPG, ODPV).",
    "version": "0.2.4",
    "auth": {
        "type": "none"
    },
    "interfaces": {
        "cli": {
            "command": "open-data-products",
            "description": "Unified command line interface for SDK workflows."
        },
        "mcp": {
            "command": "open-data-products serve",
            "description": "Safe stdio MCP server for agent hosts.",
            "transport": "stdio"
        },
        "manifest": {
            "command": "open-data-products manifest --json",
            "description": "Machine-readable capability and MCP tool manifest."
        }
    },
    "standards": [
        {
            "id": "odps",
            "name": "Open Data Product Specification",
            "description": "Data product metadata, ownership, access, SLA, and pricing."
        },
        {
            "id": "odpc",


# Lecture 8: Validating and Explaining Standards Files

Validation checks whether YAML follows an Open Data Product family standard. Explanation gives a compact human-readable view of the artifact.


### Why we validate?

Validation is the trust layer. Before we generate or combine artifacts, we need to know whether a standards file has the required structure and fields. A validation error is not just a failure; it is useful feedback that helps keep bad artifacts out of catalogs, graphs, and automation. In the SDK validation uses Open Data Products standard family specs, https://opendataproducts.org

Next, we create a valid and an intentionally invalid product file and compare the SDK feedback.

## Create Valid And Invalid ODPS Files


### Before You Run This

This lesson uses two small YAML files from the cloned course repository. One is valid, and one is intentionally schema-invalid. The invalid file is still valid YAML text; it fails because it is missing required ODPS content. That distinction matters: syntax errors and standards validation errors teach different things.

What the code does:

- Moves into the Lecture 8 folder in the cloned course repository.
- Lists the sample files so you can see the valid and invalid YAML inputs before validating them.


In [9]:
%cd /content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
!find . -maxdepth 1 -type f | sort

/content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
./invalid-product.yaml
./product.yaml
./README.md


## Validate A Correct Product


In [10]:
%cd /content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
!open-data-products validate product.yaml
!open-data-products explain product.yaml


/content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
✓ Loaded ODPS document: product.yaml
✓ Detected kind: OpenDataProduct
✓ Detected version: 4.0
✓ Schema validation passed
✓ ODPS validation passed

Validation successful!
File: product.yaml
Schema: https://opendataproducts.org/v4.0/schema/odps.json
ODPS version: 4.0
Product id: airport-operations-performance
Product name: Airport Operations Performance
Status: production
Visibility: public
Type: dataset
Components: 0
Compliance level: minimal
Production ready: False
Data access: (not set)


## Validate A Schema-Invalid Product

This file is readable YAML, but it is missing the required `productID` field.


### Before You Run This

Expect this command to report a validation problem. That is the point of the exercise. In real work, this feedback tells you what to fix before the artifact enters a catalog, graph, or automated workflow.

What the code does:

- Runs validation against the intentionally invalid product file.
- Uses `--json` so the error details are structured and easy to inspect.


In [12]:
!open-data-products validate invalid-product.yaml


✓ Loaded ODPS document: invalid-product.yaml
✓ Detected kind: OpenDataProduct
✓ Detected version: 4.0
✗ Schema validation failed

Validation failed.
✗ Validation errors: Product ID is required


## Load A Lightweight Summary

`summary` returns file-level metadata and references. It does not return the full document body.


In [13]:
!open-data-products summary product.yaml \
  --json \
  | python -m json.tool


{
    "path": "product.yaml",
    "byte_size": 416,
    "line_count": 10,
    "sha256": "4331944397379ceec1c8be22628a05fc2e92edde1a2e72b5da3cf22675f34bd7",
    "spec": "odps",
    "kind": "OpenDataProduct",
    "id": "airport-operations-performance",
    "context_artifacts": []
}


# Lecture 9: Use The ODPV Vocabulary Helpers

ODPV helpers make shared vocabulary easier to search, resolve, explain, and check.


### What We Just Covered

ODPV helps keep language consistent across product specs, catalogs, and generated outputs. Without controlled vocabulary, the same idea can appear under many labels, which makes review and automation harder.

Next, we use vocabulary helpers to summarize, search, resolve, and inspect vocabulary relationships.

## Summarize And Search The Vocabulary


### Before You Run This

Vocabulary commands are not generating new data products. They help you inspect the controlled language available in ODPV so that product specs, catalogs, and generated fragments use consistent terms.

What the code does:

- Prints a summary of the available ODPV vocabulary.
- Searches the vocabulary for terms related to governance, policy, and risk.
- Formats both results as readable JSON.


In [5]:
!open-data-products odpv-summary \
  --json \
  | python -m json.tool \
  | head -80
!open-data-products odpv-search "governance policy risk" \
  --limit 3 \
  --json \
  | python -m json.tool


{
    "valid": true,
    "errors": [],
    "term_count": 59,
    "relationship_count": 16,
    "section_count": 4,
    "spec": "odpv",
    "kind": "Vocabulary"
}
{
    "spec": "odpv",
    "kind": "VocabularyTermSearch",
    "matches": [
        {
            "score": 28,
            "matchedFields": [
                "id",
                "preferredLabel",
                "alsoKnownAs",
                "definition",
                "examples",
                "section"
            ],
            "vocabularyVersion": "1.0.0",
            "section": "governance",
            "id": "Policy",
            "uri": "https://opendataproducts.org/odpv-v1.0/terms/Policy",
            "preferredLabel": {
                "en": "Policy"
            },
            "definition": {
                "en": "A rule, guideline, or governance statement that applies to a data product, catalog, graph, use case, access method, or related object."
            },
            "relatedTerms": [
                "Com

## Resolve, Explain, And Check Relationships


### Before You Run This

This cell uses three ODPV helper commands to move from a loose phrase to a clearer vocabulary model.

What the code does:

- `odpv-resolve` takes a human phrase, `reusable data asset`, and finds the closest vocabulary concept.
- `odpv-explain` opens up the meaning of a known term, here `DataProduct`.
- `odpv-relationship` checks whether a relationship between two terms, `DataProduct supports UseCase`, is recognized by the vocabulary model.
- Each command uses `--json` and `python -m json.tool` so the structured result is easier to read in the notebook.

In [6]:
!open-data-products odpv-resolve "reusable data asset" \
  --json \
  | python -m json.tool
!open-data-products odpv-explain DataProduct \
  --json \
  | python -m json.tool
!open-data-products odpv-relationship DataProduct supports UseCase \
  --json \
  | python -m json.tool


{
    "query": "reusable data asset",
    "vocabularyVersion": "1.0.0",
    "match": {
        "section": "core",
        "id": "DataProduct",
        "uri": "https://opendataproducts.org/odpv-v1.0/terms/DataProduct",
        "type": "object",
        "status": "stable",
        "introducedIn": "1.0.0",
        "preferredLabel": {
            "en": "Data Product"
        },
        "definition": {
            "en": "A managed data offering designed for reuse, with defined ownership, access, quality, usage terms, and value context."
        },
        "alsoKnownAs": {
            "en": [
                "data product",
                "data offering",
                "reusable data asset",
                "data product asset"
            ]
        },
        "relatedTerms": [
            "Dataset",
            "DataService",
            "Distribution"
        ],
        "usedIn": [
            "ODPS",
            "ODPC",
            "ODPG"
        ],
        "examples": {
            "e

# Lecture 10: Working With Local LLMs

Local LLMs are useful for cost control, development, privacy, and restricted environments. In the main guide this lesson uses Ollama locally.

Colab is not the best place to run that part because the notebook runtime does not normally have your local Ollama server. Treat this lecture as a local-machine workflow, then use the online-provider cells below for Colab execution.


### What We Just Covered

Local LLMs can help with cost control, privacy, restricted environments, and repeatable development tests. The tradeoff is that local setup depends on the learner's machine and model runtime, so this Colab workbook treats local execution as a pattern to understand rather than a required cloud exercise.

Next, we review the local-machine workflow before moving to online provider generation.

## Local Machine Pattern

The local LLM lesson has sample files in the cloned course repo under:

```text
/content/odps-sdk-course-exercises/10-working-with-local-llms/
```

When running outside Colab, open the same lesson folder in your local clone and run the commands from inside it:

```bash
ollama pull qwen2.5
ollama list
open-data-products generate \
  --config generation.config.yaml \
  --input source_docs/turnaround-delay-signal.txt \
  --kind signal
```

The important SDK idea is the same in local and online modes: source text goes in, standards-shaped YAML comes out.

# Lecture 11: Working With Online LLM Providers

Colab works best with hosted providers. Store your API key in Colab secrets as `ANTHROPIC_API_KEY`, then load it into the notebook environment.


### What We Just Covered

Online providers let the same SDK workflow run against hosted models. The key separation is configuration versus secrets: config selects provider and model, while API keys stay in environment variables or notebook secrets.

Next, we store the provider key safely, create a generation config, and run an online generation example.

## Store The Provider Key

Do not paste real API keys into notebook cells. Use Colab secrets instead.


### Before You Run This

Hosted LLM providers need API keys. The safe pattern is to keep secrets outside YAML config files and source files. In Colab, use notebook secrets; locally, use environment variables.

What the code does:

- Tries to read `ANTHROPIC_API_KEY` from Colab secrets.
- Falls back to an existing environment variable if the notebook is not running in Colab.
- Stores the key in `os.environ` so later SDK commands can use it.
- Prints whether generation cells can run in this session.


In [25]:
import os

try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    key = os.environ.get("ANTHROPIC_API_KEY")

if key:
    os.environ["ANTHROPIC_API_KEY"] = key
    print("ANTHROPIC_API_KEY is available for this runtime.")
else:
    print("ANTHROPIC_API_KEY is not set. LLM generation cells will be skipped until you add the secret.")


ANTHROPIC_API_KEY is available for this runtime.


## Use The Lesson Generation Config

The cloned lesson folder already includes `generation.config.yaml`. The config keeps provider details separate from the command you run.


## Inspect The Generation Config

Before running generation, inspect the config file used by this lesson. This makes the provider setup visible instead of hiding it behind the command.

### Before You Run This

The generation config tells the SDK which provider and model to use, where source files live, and where generated output should be written. It should not contain the actual API key. Instead, it names the environment variable that stores the key.

What the code does:

- Moves into the Lecture 11 folder.
- Prints `generation.config.yaml` so you can see the provider settings.
- Shows the source document that will be sent into generation.
- Creates the `fragments/` output folder used by the next generation cell.

In [26]:
# Show the generation config used by this lesson.
%cd /content/odps-sdk-course-exercises/11-working-with-online-llm-providers
!cat generation.config.yaml
print("\n--- Source documents ---")
!find source_docs -maxdepth 1 -type f | sort
print("\n--- Source preview ---")
!sed -n '1,80p' source_docs/passenger-flow-product.md
!rm -rf fragments
!mkdir -p fragments


/content/odps-sdk-course-exercises/11-working-with-online-llm-providers
provider: openai
model: gpt-4.1-mini
input: source_docs/
output: fragments/

providers:
  openai:
    type: openai
    model: gpt-4.1-mini
    baseUrl: https://api.openai.com/v1
    apiKeyEnv: OPENAI_API_KEY

  claude:
    type: anthropic
    model: claude-sonnet-4-5
    baseUrl: https://api.anthropic.com/v1
    apiKeyEnv: ANTHROPIC_API_KEY
    version: "2023-06-01"
    maxTokens: 4096

--- Source documents ---
source_docs/passenger-flow-product.md

--- Source preview ---
# Passenger Flow Data Product

Airport teams need a reusable data product that combines security queue wait
time, passenger volume, boarding gate allocation, and baggage belt assignment.
The product helps terminal operations teams identify congestion before it
affects departure reliability.


## Generate One Product Reference Fragment

This command uses the source file and writes one ODPC product reference fragment.


### Before You Run This

A product reference fragment is a small portfolio object, not a full ODPS product specification. It is useful when you want a catalog-ready reference that can later be combined with objectives, use cases, signals, and graph relationships.

What the code does:

- Moves into the Lecture 11 folder.
- Checks whether the API key is available.
- Runs `generate` with the lesson config file and asks for one `product-reference` fragment.
- Lists the generated fragment files when generation succeeds.


In [27]:
%cd /content/odps-sdk-course-exercises/11-working-with-online-llm-providers

import os
from google.colab import userdata

api_key = userdata.get("ANTHROPIC_API_KEY")

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key

    !open-data-products generate \
      --config generation.config.yaml \
      --provider claude \
      --kind product-reference

    !find fragments -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to run this generation cell.")

/content/odps-sdk-course-exercises/11-working-with-online-llm-providers
Generated 1 artifact(s) in fragments/ (valid YAML)
- productReference:passenger-flow: fragments/product_reference_passenger-flow.yaml
fragments/product_reference_passenger-flow.yaml


In [29]:
%cat fragments/product_reference_passenger-flow.yaml

productReference:
  id: passenger-flow
  productID: passenger-flow
  productVersion: 1.0.0
  name:
    en: Passenger Flow Data Product
  description:
    en: Reusable data product combining security queue wait time, passenger volume,
      boarding gate allocation, and baggage belt assignment to help terminal operations
      teams identify congestion before it affects departure reliability.
  visibility: internal
  status: production
  type: dataset
  domains:
  - airport operations
  - terminal operations
  owner:
    team: Airport Data Platform Team
  productModel:
    standard: ODPS
    version: '4.1'
    format: yaml
    $ref: products/passenger-flow.yaml


# Lecture 12: Generating ODPS Data Products From Business Requirements

This lecture generates full ODPS product YAML drafts from source documents. These are data product drafts, not catalog fragments.


### What We Just Covered

Business requirements can become structured ODPS data product drafts, but generated YAML is still a draft. The useful loop is generate, inspect, improve the source or prompt if needed, and validate.

Next, we prepare source documents and generate minimal ODPS product drafts.

## Review Source Documents

Use the course sample documents directly from the cloned Lecture 12 folder.

In [25]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products
!rm -rf products
!mkdir -p products
!find source_docs -maxdepth 1 -type f | sort

/content/odps-sdk-course-exercises/12-generating-odps-data-products
source_docs/partner-api-product-brief.md
source_docs/support-operations-email-thread.md


In [26]:
%cat source_docs/partner-api-product-brief.md

# Partner Inventory Availability API

The partnerships team is planning a data-driven API product for selected retail
partners. The product will expose near-real-time inventory availability,
reserved stock, store pickup capacity, and substitution recommendations.

Audience:
- Retail partner integration teams
- Marketplace operations managers
- Internal partner success managers

Primary value:
Partners can reduce failed orders and improve customer promise accuracy by
checking availability before checkout. Marketplace operations can monitor which
partners are calling the API and whether stock availability data is current.

Access and delivery:
The first release should be an API product. Partners will authenticate with API
keys issued through the partner portal. The API returns JSON. Documentation will
be published in the partner developer portal, but the final URL is not available
yet.

Service expectations:
The API should target high availability during marketplace trading hours.
Latenc

## Generate Minimal ODPS Product Drafts

The `minimal` profile stays close to source-backed facts.


### Before You Run This

Here the target is different: `--kind odps-product` asks for full data product drafts. The `minimal` profile keeps the generated files smaller so they are easier to inspect first. Generated YAML should always be reviewed and validated before reuse.

What the code does:

- Moves into the Lecture 12 folder.
- Checks whether the API key is available.
- Runs `generate` over the `source_docs/` folder.
- Uses `--kind odps-product` and `--profile minimal` to create smaller ODPS product drafts.
- Writes generated files into `products/` and lists them.


In [28]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products
import os
from google.colab import userdata

api_key = userdata.get("ANTHROPIC_API_KEY")

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/ \
      --kind odps-product \
      --profile minimal \
      --output products/
    !find products -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to run ODPS generation.")

/content/odps-sdk-course-exercises/12-generating-odps-data-products
Generated 2 artifact(s) in products/ (valid YAML)
- odpsProduct:partner-inventory-availability-api: products/odps_product_partner-inventory-availability-api.yaml
- odpsProduct:support-operations-performance: products/odps_product_support-operations-performance.yaml
products/odps_product_partner-inventory-availability-api.yaml
products/odps_product_support-operations-performance.yaml


In [29]:
%cat products/odps_product_partner-inventory-availability-api.yaml

schema: https://opendataproducts.org/v4.1/schema/odps.json
version: '4.1'
product:
  details:
    en:
      productID: partner-inventory-availability-api
      name: Partner Inventory Availability API
      visibility: public
      status: draft
      type: data-driven service
      description: Data-driven API product that exposes near-real-time inventory availability,
        reserved stock, store pickup capacity, and substitution recommendations for
        selected retail partners
      valueProposition: Partners can reduce failed orders and improve customer promise
        accuracy by checking availability before checkout


### Support SDK development by reporting bugs!

The Data Product SDK is still maturing. If you hit bugs, please report it on GitHub

https://github.com/Open-Data-Product-Initiative/odp-agent-sdk/issues

If you can, include a minimal reproduction or a pull request with a fix. These reports help improve the SDK’s performance and reliability.  

## Optional Complete Draft Profile

Use `complete-draft` when you want the SDK to draft common review-needed ODPS components such as SLA, data quality, and pricing plans.


In [ ]:
# Optional. This can use more tokens than the minimal profile.
# Uncomment after you have reviewed the minimal output.
#
# !open-data-products generate \
#   --provider claude \
#   --model claude-sonnet-4-5 \
#   --input source_docs/ \
#   --kind odps-product \
#   --profile complete-draft \
#   --output products/


## Validate Generated Products


### Before You Run This

This validation step only has something to validate if the generation cell produced YAML files. If generation was skipped because the API key was missing, add the key, rerun generation, and then return to this cell.

What the code does:

- Moves into the Lecture 12 folder.
- Loops over every YAML file in `products/`.
- Validates each generated ODPS product draft.
- Prints a clear message if no product YAML files exist yet.

In [30]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products

from pathlib import Path

product_files = sorted(Path("products").glob("*.yaml"))
if product_files:
    for product_file in product_files:
        !open-data-products validate {product_file}
else:
    print("No generated products found. Add ANTHROPIC_API_KEY, rerun ODPS generation, then validate again.")

/content/odps-sdk-course-exercises/12-generating-odps-data-products
✓ Loaded ODPS document: products/odps_product_partner-inventory-availability-api.yaml
✓ Detected kind: OpenDataProduct
✓ Detected version: 4.1
✓ Schema validation passed
✓ ODPS validation passed

Validation successful!
✓ Loaded ODPS document: products/odps_product_support-operations-performance.yaml
✓ Detected kind: OpenDataProduct
✓ Detected version: 4.1
✓ Schema validation passed
✓ ODPS validation passed

Validation successful!


# Lecture 13: Working With Generation And Fragments

Fragments are smaller ODPC authoring units. They are useful when you want independently reviewable products, use cases, objectives, and signals before building a catalog or graph.


### What We Just Covered

Fragments are small standalone portfolio objects. They are easier to review than one large artifact and can later be assembled into catalogs and graphs. This is the bridge from one generated product draft toward portfolio-level work.

Next, we generate ODPC fragments and build a catalog from them.

## Prepare Fragment Source Folders


In [37]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
!rm -rf fragments catalog.yaml
!mkdir -p fragments
!find source_docs -type f | sort

/content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
source_docs/.DS_Store
source_docs/objectives/reduce-churn-objective.txt
source_docs/products/customer-analytics-product.md
source_docs/signals/churn-risk-signal.txt
source_docs/use-cases/customer-retention-use-case.md


## Review The Fragment Source Documents

Before generating fragments, look at the source material. Each lane contains a different type of business input, and each generate command will turn one lane into one fragment type.

### Before You Run This

This preview makes the next command set easier to understand. You are not asking the SDK to invent fragments from nowhere; you are giving it short business source documents organized by lane: products, use cases, objectives, and signals.

What the code does:

- Opens the `source_docs/` folder for Lecture 13.
- Loops through the four source lanes.
- Prints the file name and a short preview from one source document in each lane.


In [38]:
# Preview one source document from each lane before generating fragments.
from pathlib import Path

base = Path("/content/odps-sdk-course-exercises/13-working-with-generation-and-fragments/source_docs")
for lane in ["products", "use-cases", "objectives", "signals"]:
    files = sorted((base / lane).glob("*"))
    if not files:
        print(f"\n## {lane}: no files found")
        continue
    path = files[0]
    print(f"\n## {lane}: {path.name}")
    print(path.read_text()[:700].strip())



## products: customer-analytics-product.md
# Customer Analytics Product

The product provides customer profile, purchase, support, and churn risk data
for retention teams. It is used by the Customer Retention use case and supports
the Reduce Preventable Churn objective. It consumes the Churn Risk Signal.

## use-cases: customer-retention-use-case.md
# Customer Retention

Retention teams need trusted customer analytics to identify customers at risk
and choose the next best action. The use case depends on the Customer Analytics
Product, monitors the Churn Risk Signal, and contributes to the Reduce
Preventable Churn objective.

## objectives: reduce-churn-objective.txt
Objective: reduce preventable churn by improving retention decision quality and
intervention timing. The Customer Retention use case and Customer Analytics
Product support this objective. Churn Risk Signal measures early warning risk.

## signals: churn-risk-signal.txt
Daily retention briefing from April 18, 2026 at 09:30.

## Generate ODPC Fragments

Each command uses one source lane and one fragment kind.


### Before You Run This

This cell runs the same generation workflow four times, once for each source lane:

- `source_docs/products/` becomes `product-reference` fragments.
- `source_docs/use-cases/` becomes `use-case` fragments.
- `source_docs/objectives/` becomes `objective` fragments.
- `source_docs/signals/` becomes `signal` fragments.

Every command uses the same provider and model, then writes its YAML output into the shared `fragments/` folder. After the four generation commands finish, the final `find` command lists the fragment files so you can confirm what was created.

The `if os.environ.get("ANTHROPIC_API_KEY")` wrapper keeps the notebook readable even when the API key is missing. With a key, the generation runs. Without a key, the notebook prints a skip message instead of failing unexpectedly.

What the code does:

- Moves into the Lecture 13 folder.
- Checks whether the API key is available.
- Runs four `generate` commands, one per source lane.
- Uses a different `--kind` value for each lane so the SDK creates the right fragment type.
- Lists the generated YAML files in `fragments/`.


In [39]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
import os
from google.colab import userdata

api_key = userdata.get("ANTHROPIC_API_KEY")

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/products/ \
      --kind product-reference \
      --output fragments/
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/use-cases/ \
      --kind use-case \
      --output fragments/
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/objectives/ \
      --kind objective \
      --output fragments/
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/signals/ \
      --kind signal \
      --output fragments/
    !find fragments -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to generate fragments.")


/content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
Generated 1 artifact(s) in fragments/ (valid YAML)
- productReference:customer-analytics-product: fragments/product_reference_customer-analytics-product.yaml
Generated 1 artifact(s) in fragments/ (valid YAML)
- useCase:customer-retention: fragments/use_case_customer-retention.yaml
Generated 1 artifact(s) in fragments/ (valid YAML)
- businessObjective:reduce-preventable-churn: fragments/business_objective_reduce-preventable-churn.yaml
Generated 1 artifact(s) in fragments/ (valid YAML)
- signal:churn-risk-pattern-signal: fragments/signal_churn-risk-pattern-signal.yaml
fragments/business_objective_reduce-preventable-churn.yaml
fragments/product_reference_customer-analytics-product.yaml
fragments/signal_churn-risk-pattern-signal.yaml
fragments/use_case_customer-retention.yaml


## Build A Catalog From Generated Fragments

Run this after the generation cell has produced fragment YAML files.


### Before You Run This

This cell turns the generated fragment files into one ODPC catalog:

- The Python `if` checks whether `fragments/` contains any `.yaml` files.
- If fragment files exist, `odpc-build` reads the folder and assembles `catalog.yaml`.
- The validation command checks that the resulting catalog has a standards-compatible structure.
- If no fragments exist yet, the cell prints a skip message instead of failing.

This is the first composition step in the course. The previous cell created separate objects. This cell combines those objects into a portfolio-level catalog artifact.

What the code does:

- Moves into the Lecture 13 folder.
- Checks whether fragment YAML files exist before trying to build a catalog.
- Runs `odpc-build` to assemble `catalog.yaml` from the fragments.
- Validates the resulting catalog.


In [41]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
if len([p for p in __import__('pathlib').Path('fragments').glob('*.yaml')]) > 0:
    !open-data-products odpc-build fragments/ \
      --output catalog.yaml
    !open-data-products validate catalog.yaml
else:
    print("Skipped: no generated fragment YAML files found yet.")


/content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
Generated catalog.yaml (productReferences=1, useCases=1, businessObjectives=1, signals=1)
✓ Loaded ODPC document: catalog.yaml
✓ Detected kind: Catalog
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPC validation passed

Validation successful!


# Lecture 14: Working With ODPC Catalogs

ODPC catalogs collect product references, use cases, business objectives, and signals into one portfolio artifact.


### What We Just Covered

ODPC catalogs combine portfolio objects into a structured portfolio artifact. Fragments are the authoring units, the catalog is the combined structure, and the graph explains relationships.

Next, we build and render a catalog, then close Section 3 before moving to the portfolio builder workflow.

## Review Sample Fragments

In [1]:
%cd /content/odps-sdk-course-exercises/15-working-with-odpc-catalogs
!rm -rf output
!mkdir -p output
!find fragments -maxdepth 1 -type f | sort

[Errno 2] No such file or directory: '/content/odps-sdk-course-exercises/15-working-with-odpc-catalogs'
/content
find: ‘fragments’: No such file or directory


## Build And Render The Catalog


### Before You Run This

This repeats the catalog idea with ready-made fragments. The YAML file is the structured artifact; the HTML file is for human review. Both views come from the same portfolio objects.

What the code does:

- Moves into the Lecture 15 folder.
- Builds `output/catalog.yaml` from the sample fragments.
- Renders `output/catalog.html` for human review.
- Validates and summarizes the catalog YAML.


In [51]:
%cd /content/odps-sdk-course-exercises/15-working-with-odpc-catalogs
import os
from google.colab import userdata

api_key = userdata.get("ANTHROPIC_API_KEY")

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key
    !open-data-products odpc-build fragments/ \
      --output output/catalog.yaml \
      --html output/catalog.html
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to build catalog.")
!open-data-products validate output/catalog.yaml
!open-data-products odpc-summary output/catalog.yaml

/content/odps-sdk-course-exercises/15-working-with-odpc-catalogs
Generated output/catalog.yaml (productReferences=1, useCases=1, businessObjectives=1, signals=1)
✓ Loaded ODPC document: output/catalog.yaml
✓ Detected kind: Catalog
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPC validation passed

Validation successful!
File: output/catalog.yaml
Schema: https://opendataproducts.org/odpc-v1.0/schema/odpc.yaml
ODPC version: 1.0
Catalog id: CAT-GENERATED
Catalog name: Generated ODPC Catalog
Status: (not set)
Product references: 1
Use cases: 1
Business objectives: 1
Signals: 1
Graph: (not set)
Product reference ids: PR-CUSTOMER-HEALTH-SIGNALS
Use case ids: UC-RETENTION-RISK-WORKFLOW
Business objective ids: OBJ-REDUCE-CHURN
Signal ids: SIG-CHURN-DEMAND
Hints:
- No graph reference found; use Catalog.metadata.graph when relationships are implemented in ODPG or another graph standard.


## Package And Review The Catalog Output

The catalog HTML is the human review view for the ODPC catalog. This cell packages the catalog YAML and HTML together so you can inspect them outside Colab.

After the download finishes:

1. Unzip `odpc-catalog-output.zip` on your computer.
2. Open `output/catalog.html` in a browser.
3. Keep `output/catalog.yaml` beside it as the machine-readable source artifact.

The browser view helps people review the catalog, while the YAML remains the file used by validation, automation, and AI agents.

In [50]:
!zip -r odpc-catalog-output.zip output
from google.colab import files
files.download("odpc-catalog-output.zip")


  adding: output/ (stored 0%)
  adding: output/catalog.yaml (deflated 53%)
  adding: output/catalog.html (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Lecture 15: Working With ODPG Graphs

ODPG graphs make relationships between products, use cases, objectives, and signals explicit. This part uses prepared sample fragments, so it can run even if you skipped LLM generation.


### What We Just Covered

Catalogs describe portfolio objects; graphs describe relationships between those objects. In graph terms, products, use cases, objectives, and signals become nodes, while links between them become edges.

Next, we build, validate, render, and package graph output.

## Review Sample Fragments

In [42]:
%cd /content/odps-sdk-course-exercises/14-working-with-odpg-graphs
!rm -rf output
!mkdir -p output
!find fragments -maxdepth 1 -type f | sort

/content/odps-sdk-course-exercises/14-working-with-odpg-graphs
fragments/business_objective_reduce_churn.yaml
fragments/product_reference_customer_health_signals.yaml
fragments/signal_churn_demand.yaml
fragments/use_case_retention_risk_workflow.yaml


## Build, Validate, And Render The Graph


### Before You Run This

The graph is about relationships. Products, use cases, objectives, and signals become nodes; their links become edges. The rendered HTML helps humans inspect those relationships, while `graph.yaml` stays machine-readable.

This cell runs four steps:

- `odpg-build` reads the sample `fragments/` folder and creates `output/graph.yaml`.
- `validate` checks that the graph YAML follows the expected standards structure.
- `odpg-summary` prints a quick overview of the graph so you can see what was built.
- `odpg-generate` creates `output/graph-explorer.html`, a browser-friendly graph view.

The important idea is that the same relationship model has two outputs: YAML for automation and HTML for human review.

What the code does:

- Moves into the Lecture 14 folder.
- Builds a graph YAML file from the sample fragments.
- Validates and summarizes the graph.
- Renders a browser-friendly graph explorer HTML file.


In [52]:
%cd /content/odps-sdk-course-exercises/14-working-with-odpg-graphs
import os
from google.colab import userdata

api_key = userdata.get("ANTHROPIC_API_KEY")

if api_key:
    os.environ["ANTHROPIC_API_KEY"] = api_key
    !open-data-products odpg-build fragments/ \
      --provider claude \
      --model claude-sonnet-4-5 \
      --output output/graph.yaml \
      --id customer-retention-graph \
      --name "Customer Retention Graph"
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to run Graph build.")
## Note! No more LLM needed below
!open-data-products validate output/graph.yaml
!open-data-products odpg-summary output/graph.yaml
!open-data-products odpg-generate output/graph.yaml \
  --output output/graph-explorer.html


/content/odps-sdk-course-exercises/14-working-with-odpg-graphs
Generated output/graph.yaml (nodes=4, edges=5)
✓ Loaded ODPG document: output/graph.yaml
✓ Detected kind: Graph
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPG validation passed

Validation successful!
ODPG Graph: Customer Retention Graph
ID: customer-retention-graph
Nodes: 4
Edges: 5
Node types: BusinessObjective=1, DataProduct=1, Signal=1, UseCase=1
Edge types: contributesTo=1, dependsOn=1, relatedTo=1, supports=2
Graph Explorer generated successfully: output/graph-explorer.html


## Extract Agent Context Around One Node


### Before You Run This

Agent context is a focused slice of the graph around one node. Instead of giving an AI agent the whole portfolio, this command retrieves the nearby relationships that matter for a specific product, use case, or objective.

What the code does:

- Reads `output/graph.yaml`.
- Starts from the product node `PR-CUSTOMER-HEALTH-SIGNALS`.
- Retrieves nearby graph context up to depth 2.
- Formats the result as readable JSON.


In [53]:
!open-data-products odpg-agent-context output/graph.yaml \
  --node PR-CUSTOMER-HEALTH-SIGNALS \
  --depth 2 \
  --json \
  | python -m json.tool


{
    "focusNode": {
        "id": "PR-CUSTOMER-HEALTH-SIGNALS",
        "type": "DataProduct",
        "$ref": "product_reference_customer_health_signals.yaml"
    },
    "relatedNodes": [
        {
            "id": "OBJ-REDUCE-CHURN",
            "type": "BusinessObjective",
            "$ref": "business_objective_reduce_churn.yaml"
        },
        {
            "id": "PR-CUSTOMER-HEALTH-SIGNALS",
            "type": "DataProduct",
            "$ref": "product_reference_customer_health_signals.yaml"
        },
        {
            "id": "SIG-CHURN-DEMAND",
            "type": "Signal",
            "$ref": "signal_churn_demand.yaml"
        },
        {
            "id": "UC-RETENTION-RISK-WORKFLOW",
            "type": "UseCase",
            "$ref": "use_case_retention_risk_workflow.yaml"
        }
    ],
    "forwardPaths": [
        {
            "start": "PR-CUSTOMER-HEALTH-SIGNALS",
            "end": "OBJ-REDUCE-CHURN",
            "depth": 1,
            "relationships": [

## Package And Review The Graph Output

The graph explorer is an HTML file, so the important review step happens in a browser. This cell packages the whole `output/` folder and downloads it from Colab.

After the download finishes:

1. Unzip `odpg-graph-output.zip` on your computer.
2. Open `output/graph-explorer.html` in a browser.
3. Compare the visual graph with the machine-readable `output/graph.yaml`.

This reinforces the main pattern: the YAML file is for automation, and the HTML file is for human inspection.

In [54]:
!zip -r odpg-graph-output.zip output
from google.colab import files
files.download("odpg-graph-output.zip")


  adding: output/ (stored 0%)
  adding: output/graph.yaml (deflated 61%)
  adding: output/graph-explorer.html (deflated 79%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Section 3 Recap

You have now used the SDK to:

- install and inspect the CLI
- validate, explain, and summarize standards files
- use ODPV vocabulary helpers
- understand where local LLMs fit
- configure a hosted LLM provider in Colab
- generate ODPS product drafts
- generate ODPC fragments
- build and render ODPG graphs
- build and render ODPC catalogs

Section 4 builds on these pieces by turning source material into a complete portfolio workspace.


### What We Just Covered

Section 3 moved from SDK setup into the core building blocks: validation, vocabulary helpers, provider configuration, generation, fragments, graphs, and catalogs. These are the individual capabilities that the final portfolio workflow will combine.

Before continuing, this is a good point to pause for the Section 3 quiz or recap.